# La Maratona Impossibile: Lunghezza vs Abbandono 🏃‍♂️💤

**Data Storytelling a cura di AI Assistant**

---

### 📣 Messaggio Chiave
La lunghezza è un rischio, ma la qualità è l'antidoto. I 'Titani' non sopravvivono perché sono lunghi, ma perché hanno costruito un culto che inverte le regole dell'abbandono.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ast
import warnings
from solution.src.data import load_processed

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print("Libraries loaded and environment configured. ✅")

## Data Preparation & Cleaning 🛠️

Carichiamo i dati e calcoliamo le metriche fondamentali per la nostra analisi:
- **Drop Rate**: La percentuale di utenti che ha abbandonato l'anime (`dropped / members`).
- **Fasce di Lunghezza**: Categorizzazione in Short, Medium, Long, Marathon.
- **Start Year**: Per l'analisi temporale.

In [ ]:
# Load Data
df_details = load_processed("details") # Metadata (episodes, source, etc.)
df_stats = load_processed("stats")     # Stats (dropped, members, score)

# Merge
df = df_details.merge(df_stats, on="anime_id", how="inner")

# Data Cleaning & Feature Engineering
# 1. Drop Rate
df['drop_rate'] = df['dropped'] / df['members']

# 2. Clean Episodes
# Gestione episodi sconosciuti (spesso serie in corso come One Piece)
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')

# FILTRO RICHIESTO: Consideriamo solo serie con > 1 episodio (No Film/Special singoli)
# E INCLUDIAMO le serie ongoing (che hanno NaN in episodes)
df = df[(df['episodes'] > 1) | (df['episodes'].isna())]

# Per poter plottare gli ongoing, assegniamo un valore proxy (Max episodi nel dataset + bonus)
# Questo li farà apparire come "Marathon" nei grafici
max_episodes = df['episodes'].max()
if pd.isna(max_episodes): max_episodes = 1000 # Fallback safety
df['episodes'] = df['episodes'].fillna(max_episodes + 100)

df = df.dropna(subset=['drop_rate', 'score'])
df = df[df['members'] > 100] # Ignore extremely obscure anime to reduce noise

# 3. Length Classes
def classify_length(ep):
    if ep < 13: return 'Short (<13)'
    elif ep <= 26: return 'Medium (13-26)'
    elif ep <= 100: return 'Long (27-100)'
    else: return 'Marathon (>100)'

df['length_class'] = df['episodes'].apply(classify_length)
# Order categories for plots
length_order = ['Short (<13)', 'Medium (13-26)', 'Long (27-100)', 'Marathon (>100)']

# 4. Year Extraction
df['start_year'] = pd.to_datetime(df['start_date'], errors='coerce').dt.year
df['decade'] = (df['start_year'] // 10) * 10

print(f"Dataset ready: {len(df)} anime analyzed.")
df.head(3)

## Capitolo 1: Il Panorama (Analisi Univariata e Temporale) 🌄
**Obiettivo**: Mostrare com'è fatto il mondo degli anime "normali" prima di parlare di drop.

Il 90% degli anime sono sprint brevi. Ma esiste una popolazione nascosta?

In [ ]:
# 1. Istogramma Distribuzione Episodi (Scala Lineare)
fig1 = px.histogram(df[df['episodes'] < 200], x='episodes', nbins=50,
                   title="Il 90% degli anime sono sprint brevi (Distribuzione < 200 ep)",
                   labels={'episodes': 'Episodi'},
                   color_discrete_sequence=['#636EFA'])
fig1.update_layout(xaxis_title="Numero di Episodi", yaxis_title="Conteggio Anime")
fig1.show()

# 2. Istogramma Distribuzione Episodi (Scala Logaritmica)
fig2 = px.histogram(df, x='episodes', log_y=True,
                   title="Esiste una popolazione nascosta di medie e lunghe distanze (Log Scale)",
                   labels={'episodes': 'Episodi'},
                   color_discrete_sequence=['#EF553B'])
fig2.update_layout(xaxis_title="Numero di Episodi", yaxis_title="Conteggio Anime (Log)")
fig2.show()

# 3. Lineplot Temporale
decade_stats = df.groupby('decade')['episodes'].mean().reset_index()
decade_stats = decade_stats[decade_stats['decade'] >= 1960]

fig3 = px.line(decade_stats, x='decade', y='episodes', markers=True,
              title="Gli anime stanno diventando più brevi? (Media Episodi per Decennio)",
              labels={'decade': 'Decennio', 'episodes': 'Media Episodi'})
fig3.show()

## Capitolo 2: Il "Muro" (Lunghezza vs Drop Rate) 🧱
**Obiettivo**: Dimostrare la correlazione diretta tra fatica (lunghezza) e abbandono.

Più chiedi tempo, più perdi spettatori. La zona di pericolo è evidente.

In [ ]:
# 4. Scatterplot: Episodi vs Drop Rate
fig4 = px.scatter(df, x='episodes', y='drop_rate', opacity=0.3, log_x=True,
                 title="Più chiedi tempo, più perdi spettatori (Trend Generale)",
                 labels={'episodes': 'Episodi (Log)', 'drop_rate': 'Drop Rate'},
                 trendline="lowess")
fig4.show()

# 5. Barplot Categorico
length_stats = df.groupby('length_class')['drop_rate'].mean().reindex(length_order).reset_index()

fig5 = px.bar(length_stats, x='length_class', y='drop_rate',
             title="Le serie Marathon hanno un rischio di abbandono doppio (Avg Drop Rate)",
             labels={'length_class': 'Lunghezza', 'drop_rate': 'Drop Rate Medio'},
             color='length_class', color_discrete_sequence=px.colors.qualitative.Prism)
fig5.show()

# 6. Density Heatmap (Hexbin substitute)
fig6 = px.density_heatmap(df, x='episodes', y='drop_rate', log_x=True, nbinsx=30, nbinsy=30,
                         title="La Zona di Pericolo: Dove si concentrano gli abbandoni",
                         color_continuous_scale='Viridis')
fig6.show()

## Capitolo 3: L'Antidoto (Rating vs Drop Rate) 💊
**Obiettivo**: Introdurre la variabile "Qualità" (Score) per spiegare chi si salva.

Se piace, non si molla. L'eccellenza garantisce fedeltà.

In [ ]:
# 7. Scatterplot: Score vs Drop Rate
fig7 = px.scatter(df, x='score', y='drop_rate', opacity=0.3,
                 title="Se piace, non si molla: La qualità protegge dall'abbandono",
                 labels={'score': 'Score', 'drop_rate': 'Drop Rate'},
                 trendline="lowess", trendline_color_override="red")
fig7.show()

# 8. Boxplot per Fasce di Voto
def classify_score(s):
    if s < 6: return 'Low (<6)'
    elif s <= 7.5: return 'Mid (6-7.5)'
    else: return 'High (>7.5)'

df['score_class'] = df['score'].apply(classify_score)
score_order = ['Low (<6)', 'Mid (6-7.5)', 'High (>7.5)']

fig8 = px.box(df, x='score_class', y='drop_rate', category_orders={'score_class': score_order},
             title="L'eccellenza garantisce fedeltà (Drop Rate per Fascia di Voto)",
             color='score_class')
fig8.show()

# 9. Bubble Chart Multivariato
fig9 = px.scatter(df[df['episodes'] > 10], x='episodes', y='drop_rate', 
                 size='members', color='score', log_x=True,
                 hover_name='title', size_max=40,
                 title="Il Quadro Completo: Episodi, Drop, Qualità e Popolarità",
                 color_continuous_scale='RdYlGn')
fig9.show()

## Capitolo 4: Analisi per Genere (Il Pubblico conta) 🎭
**Obiettivo**: Vedere se certi fan sono più pazienti di altri.

I fan delle commedie abbandonano più facilmente dei fan dei drama? Non tutti i generi corrono la stessa gara.

In [ ]:
# Helper per gestire i generi (sia stringhe che liste)
def parse_genres(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return []

# Applica la funzione a tutta la colonna genres
# Usiamo list comprehension per evitare TypeError: unhashable type: 'list' di pandas
df['genres_list'] = [parse_genres(x) for x in df['genres']]

df_exploded = df.explode('genres_list')
top_genres = df_exploded['genres_list'].value_counts().head(10).index.tolist()
df_genres = df_exploded[df_exploded['genres_list'].isin(top_genres)]

# 10. Barplot: Lunghezza Media per Top 10 Generi
genre_len = df_genres.groupby('genres_list')['episodes'].mean().sort_values(ascending=False).reset_index()

fig10 = px.bar(genre_len, x='genres_list', y='episodes',
              title="Non tutti i generi corrono la stessa gara (Durata Media)",
              color='episodes', color_continuous_scale='Blues')
fig10.show()

# 11. Strip Plot: Drop Rate per Genere
fig11 = px.strip(df_genres, x='genres_list', y='drop_rate', color='genres_list',
                title="Dispersione dell'abbandono per Genere",
                labels={'genres_list': 'Genere'})
fig11.update_traces(jitter=1.0)
fig11.show()

# 12. Heatmap Correlazione Genere
corrs = []
for g in top_genres:
    sub = df_genres[df_genres['genres_list'] == g]
    corr = sub['episodes'].corr(sub['drop_rate'])
    corrs.append({'Genere': g, 'Corr (Len vs Drop)': corr})

corr_df = pd.DataFrame(corrs).set_index('Genere').sort_values('Corr (Len vs Drop)', ascending=False)

fig12 = px.imshow(corr_df.T,
                 title="Dove la lunghezza fa più male? (Correlazione Episodi-Drop)",
                 color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
fig12.show()

## Capitolo 5: I Titani e Considerazioni Finali 🗽
**Obiettivo**: L'analisi degli Outlier (One Piece, Naruto, ecc.) e sintesi.

Guardate dove sono i Titani: sono lunghissimi, ma il loro drop rate è basso come quello delle serie brevi. Sono anomalie statistiche.

In [ ]:
# Identifica i Titani (Top 20 lunghi con membri > 100k)
titans_mask = (df['episodes'] > 100) & (df['members'] > 100000)
df['type_label'] = 'Standard'
df.loc[titans_mask, 'type_label'] = 'Titan'

titans = df[titans_mask].sort_values('episodes', ascending=False).head(10)

# 13. Scatterplot con Highlighting
fig13 = px.scatter(df, x='episodes', y='drop_rate', color='type_label', 
                  log_x=True, opacity=0.4,
                  color_discrete_map={'Standard': 'lightgray', 'Titan': 'red'},
                  title="I Titani: Anomalie statistiche che sfidano la gravità",
                  hover_name='title')

# Annotazioni per i big
for i, row in titans.iterrows():
    fig13.add_annotation(x=np.log10(row['episodes']), y=row['drop_rate'],
                        text=row['title'], showarrow=True, arrowhead=1)

fig13.show()

# 14. Barplot Confronto: Titani vs Resto del Mondo (>100 ep)
long_others = df[(df['episodes'] > 100) & (~titans_mask)]
big_3 = df[df['title'].isin(['One Piece', 'Naruto', 'Bleach'])] # Esempio Big 3

comp_data = pd.DataFrame({
    'Group': ['Anime Lunghi Sconosciuti', 'Big 3 (Titani)'],
    'Drop Rate': [long_others['drop_rate'].mean(), big_3['drop_rate'].mean()]
})

fig14 = px.bar(comp_data, x='Group', y='Drop Rate',
              title="Non è la lunghezza a salvare i Titani, è il brand",
              color='Group')
fig14.show()

# 15. Matrice di Correlazione Finale
final_vars = ['episodes', 'score', 'members', 'drop_rate']
corr_matrix = df[final_vars].corr()

fig15 = px.imshow(corr_matrix, text_auto=True,
                 title="Sintesi Matematica: Lunghezza penalizza, Qualità salva",
                 color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
fig15.show()

### Conclusioni 🔚

In conclusione, i dati mostrano che la **lunghezza è un fattore di rischio significativo** per l'abbandono (correlazione positiva), specialmente per le serie di fascia media (26-50 episodi) che non riescono a giustificare il loro impegno temporale.

Tuttavia, i **'Titani'** dimostrano che un punteggio elevato (*Proxy di qualità*) e una community vasta (*Social Proof*) possono invertire questa tendenza. Non è la brevità a salvare un anime, ma la capacità di mantenere la promessa di qualità nel tempo.